# 第 11 章 · 流式输出与服务化

> **你已会**：Runner、事件流、Session、`adk api_server`（第 2 章）。
>
> **读完后你能**：① 区分事件级流式和 token 级流式；② 用 `RunConfig` 打开 token 流；③ 用 `/run_sse` 提供服务；④ 用 SQLite 让服务重启后继续对话；⑤ 写出一份可执行的上线清单。
>
> **本章不碰**：语音/实时双向流（需要额外模型能力）、云端托管部署。

**怎么读**

- **主线（约 40 分钟）**：§1 两种流式 → §2 token 流实验 → §3 服务化。
- **深读**：§4 SQLite 会话、§5 上线清单。
- 本章的代码会真实调用模型；服务命令需要你另开一个终端。

---

## 1. 两种"流式"，不要混为一谈

| 类型 | 你看到什么 | 谁产生 | 典型用途 |
|---|---|---|---|
| **事件级流式** | 每个 `Event` 一条 | `run_async()` 本来就是这样 | UI 显示"正在调用工具" |
| **token 级流式** | 一段文本被拆成多个 `partial` 事件 | 模型 API 的流式响应 | 逐字打字机效果 |

> 📌 **默认是事件级，不是 token 级。** 不开 `StreamingMode.SSE` 时，模型的整段回复会作为一个事件到达。

**先猜：一次普通回答在 token 级流式下会产生几条事件？1 条、10 条，还是接近 100 条？**

---

## 2. 打开 token 级流式

`RunConfig(streaming_mode=StreamingMode.SSE)` 告诉 Runner 使用模型的流式响应：

In [1]:
import os
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

from google.adk.agents import Agent
from google.adk.agents.run_config import RunConfig, StreamingMode
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch11", "student"

stream_agent = Agent(
    name="streamer",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="用三句话介绍 ADK。",
    description="流式输出演示",
)

service = InMemorySessionService()
await service.create_session(app_name=APP, user_id=USER, session_id="stream_demo")
runner = Runner(agent=stream_agent, app_name=APP, session_service=service)

message = types.Content(role="user", parts=[types.Part(text="介绍一下 ADK")])

events = 0
partials = 0
text = ""
async for event in runner.run_async(
    user_id=USER,
    session_id="stream_demo",
    new_message=message,
    run_config=RunConfig(streaming_mode=StreamingMode.SSE),
):
    events += 1
    if event.partial:
        partials += 1
    if event.content:
        for part in event.content.parts:
            if part.text:
                text += part.text

print("事件总数：", events)
print("partial 事件：", partials)
print("拼接后的文本长度：", len(text))

assert partials > 10, "没有收到 token 级流式事件"
assert text.strip(), "没有收到文本"
print("✅ token 级流式生效")

事件总数： 86
partial 事件： 85
拼接后的文本长度： 348
✅ token 级流式生效


> ⚠️ **不要把 partial 文本直接入库。** 每个 partial 只是"到目前为止的一段"，重复落库会得到几十条残缺记录。**只在最终事件（`partial=False`）落库。**
>
> 📌 **流式不等于更快。** 总生成时间基本不变；变的是"用户多快看到第一个字"。

---

## 3. 用 `/run_sse` 提供服务

`adk api_server` 同时暴露两个接口：

| 接口 | 返回 | 适合 |
|---|---|---|
| `POST /run` | 一次性返回事件列表 | 脚本、批处理 |
| `POST /run_sse` | `text/event-stream`，逐条推送 | 浏览器、实时 UI |

启动服务（另开终端，在仓库根目录）：

```bash
uv run adk api_server google-adk --host 127.0.0.1 --port 8000
```

先建会话：

```bash
curl -X POST http://127.0.0.1:8000/apps/agent_demo/users/student/sessions \
  -H "Content-Type: application/json" -d '{"session_id": "s1"}'
```

再请求流式接口。**关键是 `"streaming": true`**：

```bash
curl -N -X POST http://127.0.0.1:8000/run_sse \
  -H "Content-Type: application/json" \
  -d '{"app_name":"agent_demo","user_id":"student","session_id":"s1","streaming":true,
       "new_message":{"role":"user","parts":[{"text":"用一句话介绍 ADK"}]}}'
```

返回是一串 `data: {...}`。每个 `data` 都是一个事件：

- `"partial": true`：token 片段；
- `"partial"` 缺失或为 `false`：完整事件；
- `"author"`：哪个 Agent 产生的；
- `"nodeInfo.path"`：执行路径。

> ⚠️ **不传 `streaming: true` 时，`/run_sse` 仍然可用，但你拿到的是完整事件，不是逐字片段。** 这是最常见的"为什么没有打字机效果"。

---

## 4. 让服务重启后还能继续：SQLite 会话

默认的本地存储会在重启后丢失会话。用 `--session_service_uri` 指向 SQLite：

```bash
uv run adk api_server google-adk --host 127.0.0.1 --port 8000 \
  --session_service_uri="sqlite+aiosqlite:///./adk_sessions.db"
```

验证方式：

1. 建会话 `s1`，发一轮消息；
2. `Ctrl+C` 停止服务；
3. 用同一条命令重启；
4. 再向 `s1` 发消息，Agent 仍能看到上一轮历史。

> 📌 **路径是相对于启动目录的。** 在 Windows 上不要用 Git Bash 的 `/d/...` 路径拼 SQLite URL，直接用相对路径最稳。

---

## 5. 上线清单

| 维度 | 最低要求 | 常见坑 |
|---|---|---|
| 密钥 | 环境变量或 Secret Manager 注入 | 把密钥写进镜像或日志 |
| 会话 | 持久化后端 + 备份 | 重启后用户"失忆" |
| 并发 | 全局限流 + 超时 | 只设了单次请求超时 |
| 健康检查 | `/` 或独立探针 | 进程活着但模型不可用 |
| 日志 | trace/session id、错误分类 | 记录完整敏感输入 |
| 回滚 | 镜像 + 配置版本化 | 只回滚代码，没回滚 prompt |
| 评估 | 发布前跑门禁（第 10 章） | 用"手工试了一下"代替评估 |

镜像示例见 [Dockerfile](Dockerfile)：

```bash
docker build -f google-adk/Dockerfile -t adk-agent-demo .
docker run --rm -p 8000:8000 -e DEEPSEEK_API_KEY="$DEEPSEEK_API_KEY" adk-agent-demo
```

> ⚠️ **本文不验证 Docker 构建**（环境可能没有 Docker）。构建前先确认 `uv.lock` 与 `pyproject.toml` 已提交，否则镜像不可复现。

---

## 6. 流式服务的三个边界

1. **断线**：客户端断开后，服务端可能继续生成。要显式取消，避免白花 token；
2. **背压**：慢客户端会拖住连接。生产环境要限制单连接时长和并发；
3. **一致性**：partial 事件不是最终状态。入库、计费、评估都应以最终事件为准。

---

## 📌 用五句话复述本章

1. **事件级流式是默认，token 级流式要显式打开 `StreamingMode.SSE`。**
2. **`/run` 返回事件列表，`/run_sse` 逐条推送；token 级还需要请求体里的 `"streaming": true`。**
3. **partial 事件只用于显示，不能直接入库或计费。**
4. **`--session_service_uri` 指向 SQLite，服务重启后会话仍在。**
5. **上线清单里，密钥、会话、限流、健康检查、回滚、评估缺一不可。**

---

## 🧪 本章练习

### 1. 对比两种流式（基础）

- [ ] 不开 `StreamingMode` 跑一次，记录事件数
- [ ] 打开后再跑一次，记录 partial 事件数
- [ ] 用一句话解释差异

### 2. 消费 `/run_sse`（进阶）

- [ ] 启动 `adk api_server`
- [ ] 建会话并发送 `streaming: true` 请求
- [ ] 统计 `data:` 行数，确认远大于 1
- [ ] 只在最后一个事件落库

### 3. 重启后继续对话（工程）

- [ ] 用 SQLite 会话启动服务
- [ ] 发一轮消息后停止服务
- [ ] 重启后继续同一会话
- [ ] 确认历史仍在

### 4. 写一份上线检查表（综合）

- [ ] 覆盖 §5 的七个维度
- [ ] 每项写出验证命令或检查方法
- [ ] 标出哪一项失败时必须阻断发布

参考实现：[practice/c11_1.py](practice/c11_1.py)。